In [1]:
%pip install duckdb

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:

import duckdb
import pandas as pd
import pytz
import os
from datetime import datetime

# Timezone
utc_plus_8 = pytz.timezone('Australia/Perth')

# Path to DuckDB file
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
duckdb_path = os.path.join(project_root, "duckdb", "warehouse.duckdb")

# Connect to DuckDB
con = duckdb.connect(duckdb_path)

# List tables in raw_dataset
raw_tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'raw_dataset'
""").fetchdf()
print("Tables in 'raw_dataset':")
print(raw_tables)

# Sample query
try:
    df_customers = con.execute("SELECT * FROM raw_dataset.customers LIMIT 10").fetchdf()
    print("\nSample from raw_dataset.customers:")
    print(df_customers)
except Exception as e:
    print("\nError querying customers:", e)

# List tables in main_gold
main_gold_tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main_gold'
""").fetchdf()
print("\nTables in 'main_gold':")
print(main_gold_tables)

# List all schemas
schemas = con.execute("""
    SELECT schema_name
    FROM information_schema.schemata
""").fetchdf()
print("\nSchemas in database:")
print(schemas)

# Query dim_date
try:
    dim_date = con.execute("SELECT * FROM main_gold.dim_date LIMIT 10").fetchdf()
    print("\nSample from main_gold.dim_date:")
    print(dim_date)
except Exception as e:
    display("\nError querying dim_date:", e)


Tables in 'raw_dataset':
             table_name
0             customers
1                events
2         exchangerates
3          ordersheader
4           orderslines
5              products
6               sensors
7             shipments
8                stores
9             suppliers
10           _dlt_loads
11  _dlt_pipeline_state
12         _dlt_version

Sample from raw_dataset.customers:
   customer_id    natural_key   first_name  last_name  \
0            1  CUST-8RIGHPOB          Amy     Harper   
1            2  CUST-RJV1NVAK        Henry       Hess   
2            3  CUST-OOXSFECM        Holly       Park   
3            4  CUST-9O3DYKUR         Jose  Patterson   
4            5  CUST-Q93JQIP8        James   Mcdaniel   
5            6  CUST-R8A79Q3H     Veronica      Mayer   
6            7  CUST-K687JMXT      Bradley      Gomez   
7            8  CUST-458EI9KI       Donald   Campbell   
8            9  CUST-A9OBOEVE  Christopher    Shelton   
9           10  CUST-GMG10W10    

In [3]:
# Query dim_date
try:
    dim_date = con.execute("SELECT * FROM main_gold.dim_date LIMIT 10").fetchdf()
    print("\nSample from main_gold.dim_date:")
    print(dim_date)
except Exception as e:
    display("\nError querying dim_date:", e)



Sample from main_gold.dim_date:
        date  year  month  day  day_of_week   day_name month_name  quarter  \
0 2021-01-01  2021      1    1            5     Friday    January        1   
1 2021-01-02  2021      1    2            6   Saturday    January        1   
2 2021-01-03  2021      1    3            0     Sunday    January        1   
3 2021-01-04  2021      1    4            1     Monday    January        1   
4 2021-01-05  2021      1    5            2    Tuesday    January        1   
5 2021-01-06  2021      1    6            3  Wednesday    January        1   
6 2021-01-07  2021      1    7            4   Thursday    January        1   
7 2021-01-08  2021      1    8            5     Friday    January        1   
8 2021-01-09  2021      1    9            6   Saturday    January        1   
9 2021-01-10  2021      1   10            0     Sunday    January        1   

   is_weekend month_start_date  ... quarter_start_date quarter_end_date  \
0       False       2021-01-01  .

In [4]:
# List all tables across all schemas
all_tables = con.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_type = 'BASE TABLE'
    ORDER BY table_schema, table_name
""").fetchdf()

print("\nAll tables in the database:")
print(all_tables)


All tables in the database:
   table_schema            table_name
0     main_gold          dim_customer
1     main_gold              dim_date
2     main_gold       dim_product_scd
3     main_gold             dim_store
4     main_gold          dim_supplier
5   main_silver             fct_sales
6   main_silver          silver_audit
7   main_silver      silver_customers
8   main_silver         silver_events
9   main_silver  silver_exchangerates
10  main_silver      silver_fct_sales
11  main_silver         silver_orders
12  main_silver    silver_orderslines
13  main_silver       silver_products
14  main_silver        silver_sensors
15  main_silver      silver_shipments
16  main_silver         silver_stores
17  main_silver      silver_suppliers
18     main_stg         stg_countries
19     main_stg         stg_customers
20     main_stg            stg_events
21     main_stg     stg_exchangerates
22     main_stg         stg_holidayau
23     main_stg      stg_ordersheader
24     main_stg      

In [7]:
# Query dim_date
try:
    dim_date = con.execute("SELECT * FROM main_silver.silver_audit LIMIT 10").fetchdf()
    print("\nSample from main_silver.silver_audit:")
    print(dim_date)
except Exception as e:
    display("\nError querying dim_date:", e)



Sample from main_silver.silver_audit:
  status                                             timing  \
0   pass  [{'completed_at': '2025-10-07T04:24:50.312401Z...   
1   pass  [{'completed_at': '2025-10-07T04:24:50.297676Z...   
2   pass  [{'completed_at': '2025-10-07T04:24:50.317430Z...   
3   pass  [{'completed_at': '2025-10-07T04:24:50.316996Z...   
4   pass  [{'completed_at': '2025-10-07T04:24:50.360743Z...   
5   pass  [{'completed_at': '2025-10-07T04:24:50.370210Z...   
6   pass  [{'completed_at': '2025-10-07T04:24:50.369682Z...   
7   pass  [{'completed_at': '2025-10-07T04:24:50.364675Z...   
8   pass  [{'completed_at': '2025-10-07T04:24:50.426493Z...   
9   pass  [{'completed_at': '2025-10-07T04:24:50.421759Z...   

  compile_started_at compile_completed_at             execute_started_at  \
0               None                 None  "2025-10-07T04:24:50.282805Z"   
1               None                 None  "2025-10-07T04:24:50.272184Z"   
2               None                 No